In [1]:
import sys
sys.path.append('../d03_src/')
import process_infutor

# Processing Infutor

In this notebook, we walk through the main functions we use to process the Infutor records into addresses. Because the raw data is proprietary, we cannot give real-world examples. The goal of this notebook is to show how to use each of our data processing functions, so that other users with access to Infutor data can reproduce our pipeline and anyone can undersand our data cleaning process.

## 1. INFUTOR Individual Address Histories

We start with a dataframe containing individual address histories. Many users receive the Infutor data in the form of tab-separated files according to most recent state---our dataframe assumes these are all already merged (see the scripts on `d04_scripts/d01_read-files/` for any questions on this process).

The data types and columns used in the dataframe are detailed in `d04_scripts/d03_process-flows/01_read-INFUTOR-individuals.py`. We also chunk the dataframe to improve efficiency, a process documented under `d04_scripts/d03_process-flows/02_split-INFUTOR-individuals.py`.

## 2. Cleaning Address Histories

The process of cleaning address histories is explained in detail in our paper, section M.1. Here, we walk through the functions we used. The full process is documented under `d04_scripts/d03_process-flows/03_process-INFUTOR-individuals.py` and `d04_scripts/d03_process-flows/04_create-yearly-matrix.py`

### 2a. Producing a distribution of individual ACS responses

We individually process each row of the Infutor dataframe into expected ACS responses. The function `process_individual` is part of the `process_infutor` module:

In [2]:
process_infutor.process_individual?

Signature:
process_infutor.process_individual(
    individual,
    PO_box_window_in_days=365,
    pad_ends_in_days=365,
    pad_only_if_alive=False,
    min_year=1902,
    max_year=inf,
    valid_address_types=['clean', 'rural_route', 'incomplete'],
    date_columns=['IDATE', 'ODATE'],
    effdate_columns=['EFFDATE1', 'EFFDATE2', 'EFFDATE3', 'EFFDATE4', 'EFFDATE5', 'EFFDATE6', 'EFFDATE7', 'EFFDATE8', 'EFFDATE9', 'EFFDATE10'],
    address_columns=['ADDRID1', 'ADDRID2', 'ADDRID3', 'ADDRID4', 'ADDRID5', 'ADDRID6', 'ADDRID7', 'ADDRID8', 'ADDRID9', 'ADDRID10'],
    category_columns=['ADDRCAT1', 'ADDRCAT2', 'ADDRCAT3', 'ADDRCAT4', 'ADDRCAT5', 'ADDRCAT6', 'ADDRCAT7', 'ADDRCAT8', 'ADDRCAT9', 'ADDRCAT10'],
    verbose=False,
)
Docstring:
Function to process one row (individual) in the INFUTOR PID dataframe into
    an individual "distribution of ACS yearly responses" for every year in
    the range. That is, estimated yearly personal flows between ADDRIDs.
    
Returns
----------
list
    entri

This function goes through the following main subroutines:

- We clean start and end dates by imputing missing values and ensuring the dates are consistent with the address histories.

In [3]:
process_infutor.hygienize_IO_DATES?

Signature:
process_infutor.hygienize_IO_DATES(
    individual,
    date_columns=['IDATE', 'ODATE'],
    effdate_columns=['EFFDATE1', 'EFFDATE2', 'EFFDATE3', 'EFFDATE4', 'EFFDATE5', 'EFFDATE6', 'EFFDATE7', 'EFFDATE8', 'EFFDATE9', 'EFFDATE10'],
)
Docstring:
Resolve IDATE (first date a person is seen) and ODATE
    (last date a person is seen) inconsistencies.
Assigns to IDATE the earliest across all dates and 
    to ODATE the latest across all dates.

Parameters
----------
individual : pd.Series
    row of an individual DataFrame containing EFFDATE and ADDRID
    columns, as well as IDATE, ODATE, DeceasedCD
    
Returns
----------
(new_start_date, new_end_date)       
File:      /share/pierson/gs665/migration_flows/MIGRATE/d03_src/process_infutor.py
Type:      function

- We remove invalid addresses

In [4]:
process_infutor.remove_NaT_addresses?

Signature:
process_infutor.remove_NaT_addresses(
    individual,
    date_columns=['IDATE', 'ODATE'],
    effdate_columns=['EFFDATE1', 'EFFDATE2', 'EFFDATE3', 'EFFDATE4', 'EFFDATE5', 'EFFDATE6', 'EFFDATE7', 'EFFDATE8', 'EFFDATE9', 'EFFDATE10'],
    address_columns=['ADDRID1', 'ADDRID2', 'ADDRID3', 'ADDRID4', 'ADDRID5', 'ADDRID6', 'ADDRID7', 'ADDRID8', 'ADDRID9', 'ADDRID10'],
    category_columns=['ADDRCAT1', 'ADDRCAT2', 'ADDRCAT3', 'ADDRCAT4', 'ADDRCAT5', 'ADDRCAT6', 'ADDRCAT7', 'ADDRCAT8', 'ADDRCAT9', 'ADDRCAT10'],
    verbose=False,
)
Docstring:
Remove addresses without a date.

Parameters
----------
individual : pd.Series
    row of an individual DataFrame containing EFFDATE and ADDRID
    columns, as well as IDATE, ODATE, DeceasedCD
    
Returns
----------
pd.Series with EFFDATE ADDRID and ADDRCAT columns
File:      /share/pierson/gs665/migration_flows/MIGRATE/d03_src/process_infutor.py
Type:      function

- We remove PO boxes, unless they are the only available address within a given window (1 year).

In [5]:
process_infutor.remove_PO_box?

Signature:
process_infutor.remove_PO_box(
    individual,
    window_in_days=365,
    valid_address_types=['clean', 'rural_route', 'incomplete'],
    effdate_columns=['EFFDATE1', 'EFFDATE2', 'EFFDATE3', 'EFFDATE4', 'EFFDATE5', 'EFFDATE6', 'EFFDATE7', 'EFFDATE8', 'EFFDATE9', 'EFFDATE10'],
    address_columns=['ADDRID1', 'ADDRID2', 'ADDRID3', 'ADDRID4', 'ADDRID5', 'ADDRID6', 'ADDRID7', 'ADDRID8', 'ADDRID9', 'ADDRID10'],
    category_columns=['ADDRCAT1', 'ADDRCAT2', 'ADDRCAT3', 'ADDRCAT4', 'ADDRCAT5', 'ADDRCAT6', 'ADDRCAT7', 'ADDRCAT8', 'ADDRCAT9', 'ADDRCAT10'],
)
Docstring:
Remove PO Box addresses within a reasonable window of a clean
    address

Parameters
----------
individual : pd.Series
    row of an individual DataFrame containing EFFDATE, ADDRID, and
    ADDRCAT columns
    
Returns
----------
pd.Series with EFFDATE, ADDRID, and ADDRCAT columns
File:      /share/pierson/gs665/migration_flows/MIGRATE/d03_src/process_infutor.py
Type:      function

- We process the individual address history into a sequence of monthly addresses.

In [6]:
process_infutor.get_monthly_addresses?

Signature:
process_infutor.get_monthly_addresses(
    individual,
    pad_ends_in_days=365,
    pad_only_if_alive=True,
    effdate_columns=['EFFDATE1', 'EFFDATE2', 'EFFDATE3', 'EFFDATE4', 'EFFDATE5', 'EFFDATE6', 'EFFDATE7', 'EFFDATE8', 'EFFDATE9', 'EFFDATE10'],
    address_columns=['ADDRID1', 'ADDRID2', 'ADDRID3', 'ADDRID4', 'ADDRID5', 'ADDRID6', 'ADDRID7', 'ADDRID8', 'ADDRID9', 'ADDRID10'],
    category_columns=['ADDRCAT1', 'ADDRCAT2', 'ADDRCAT3', 'ADDRCAT4', 'ADDRCAT5', 'ADDRCAT6', 'ADDRCAT7', 'ADDRCAT8', 'ADDRCAT9', 'ADDRCAT10'],
)
Docstring:
Get the monthly addresses (and associated probabilities) of each individual
    during their active INFUTOR time.
    
Returns
----------
dict
    keys are months, items are dictionaries with keys
    ADDRID (list) and p (float)
File:      /share/pierson/gs665/migration_flows/MIGRATE/d03_src/process_infutor.py
Type:      function

- We map year-to-year monthly addresses to produce an estimated ACS response by the individual.

In [7]:
process_infutor.get_yearly_ACS_responses?

Signature:
process_infutor.get_yearly_ACS_responses(
    address_history,
    min_year=0,
    max_year=inf,
)
Docstring:
For an address history, get the yearly ACS responses (estimated)

Returns
----------
list
    entries are 4-tuples of the format
    
    (YEAR, ORIGIN_ADDRID, DEST_ADDRID, PROBABILITY)

    Selecting all entries of the year and mapping IDs to indices
    will generate a COO sparse matrix. Note that for the first and
    last years when an individual is active the probabilities may not
    sum to 1.
File:      /share/pierson/gs665/migration_flows/MIGRATE/d03_src/process_infutor.py
Type:      function

### 2b. Producing Aggregated ACS Responses

Now we aggregate individual responses into population-wide estimates of ACS responses to the question ``where did you live 1 year ago?``

In [8]:
process_infutor.aggregate_individual_responses?

Signature:
process_infutor.aggregate_individual_responses(
    individual_responses,
    verbose=False,
)
Docstring:
Takes a series of individual responses an aggregates into a dataframe
    of population ACS responses

Returns
----------
pd.Series
    columns are `year`, `origin`, `destination`, and `flow`
    
    `year`: year when move would be reported i.e. when ACS would be answered
    `origin` and `destination`: ADDRID
    `flow`: expected flow for the whole population (i.e. aggregates individual probabilities)
File:      /share/pierson/gs665/migration_flows/MIGRATE/d03_src/process_infutor.py
Type:      function

We then save the responses by year.

### 2c. Producing an Yearly Flow Matrix

Our next step is to create a yearly flow matrix between addresses. This processes is documented on the script `d04_scripts/d03_process-flows/04_create-yearly-matrix.py`. It entails grouping the entries of the `aggregated_individual_responses` return by `origin, destination` pair and defining a `scipy.sparse.COO_matrix` $A$

### 2d. Geocoding the Flow Matrix

Finally, we use our geocoding matrix $\mathcal{G}$ obtained via `d04_scripts/d02_process-addresses/08_produce-geocoding-matrix.py` to geocode the address-to-address matrix obtained above. This is documented in `d04_scripts/d03_process-flows/05_multiply-yearly-matrix.py` and, as detailed in the Methods M1 section of our paper, includes a diagonal adjustment to avoid double counting folks who stay on the same address:


$$E = \mathcal G ^T \cdot \left[A^{(t)} - \text{diag}\left(A^{(t)}\right)\right] \cdot \mathcal G + \text{diag}\left[\mathcal G ^T\cdot \text{diag}\left(A^{(t)}\right)\right]$$